# PIPELINE THỰC THI TOÀN DIỆN ALL-IN-ONE IMAGE RESTORATION (AiOIR)
**Môi trường thực thi**: Google Colab Free (GPU Tesla T4 ~15GB VRAM, RAM ~12GB)
**Mục tiêu thí nghiệm (1-Click Run)**:
1. **P0 (Setup & Verification)**: Khởi tạo phần cứng T4, kiểm tra AMP, cài đặt thư viện, tải và kiểm tra số lượng ảnh benchmark (BSD68, Kodak24, Urban100, Rain100L).
2. **P1 (Baseline Reproduction & Efficiency)**: Đánh giá checkpoint All-in-One PromptIR trên tập test (Denoise & Derain); đo tham số, FLOPs, độ trễ và VRAM đỉnh trên T4.
3. **P2 (Failure Analysis & OOD Diagnostic)**: Chẩn đoán 30 ảnh tệ nhất mỗi tập, phân tích miền tần số 2D FFT, quang phổ sai số và kiểm tra độ bền OOD (Out-of-Distribution).
4. **Tự động đóng gói và sao lưu**: Lưu toàn bộ logs, bảng kết quả CSV và hình ảnh trực quan hóa.

---
## BƯỚC 1: KIỂM TRA PHẦN CỨNG GPU T4 & KẾT NỐI GOOGLE DRIVE (TÙY CHỌN)

In [ ]:
# 1. Kết nối Google Drive (Tùy chọn: có thể bỏ qua nếu muốn chạy trực tiếp trên SSD Colab)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("[+] Đã kết nối Google Drive thành công!")
except Exception as e:
    print("[*] Tiếp tục chạy trên SSD cục bộ của Colab (không mount Drive).")

# 2. Kiểm tra thông số GPU T4 qua nvidia-smi
!nvidia-smi

import torch
print("=" * 60)
print("PyTorch Version :", torch.__version__)
print("CUDA Available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name :", torch.cuda.get_device_name(0))
    print("VRAM Dung lượng :", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")
print("=" * 60)

---
## BƯỚC 2: TỰ ĐỘNG KHỞI TẠO WORKSPACE /content/repo & CÀI ĐẶT MÔI TRƯỜNG

In [ ]:
import os, sys, shutil
from pathlib import Path

REPO_DIR = Path("/content/repo")

# Đảm bảo /content/repo luôn tồn tại với đầy đủ mã nguồn mới nhất từ GitHub
if not (REPO_DIR / "scripts" / "download_datasets.py").exists():
    print("Đang tải toàn bộ mã nguồn dự án từ GitHub (nthmaiii/all-in-one-image-restoration)...")
    if Path("/content/temp_repo").exists():
        shutil.rmtree("/content/temp_repo")
    !git clone https://github.com/nthmaiii/all-in-one-image-restoration.git /content/temp_repo
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    shutil.copytree("/content/temp_repo/repo", REPO_DIR)
    shutil.rmtree("/content/temp_repo")
    print(">> Khởi tạo /content/repo thành công!")
else:
    print(">> Đã có /content/repo. Đang đồng bộ scripts mới nhất từ GitHub...")
    !git clone --depth 1 https://github.com/nthmaiii/all-in-one-image-restoration.git /content/temp_update
    !cp -rf /content/temp_update/repo/scripts/* /content/repo/scripts/
    !rm -rf /content/temp_update
    print(">> Đồng bộ scripts thành công!")

%cd /content/repo

# Cài đặt thư viện theo requirements.txt
!pip install -q -r requirements.txt
!pip install -q gdown ptflops lpips
print(">> Hoàn tất cài đặt các thư viện cần thiết!")

---
## BƯỚC 3: TẢI TỰ ĐỘNG DỮ LIỆU BENCHMARK VÀ CHECKPOINT TỪ TÁC GIẢ GỐC

In [ ]:
%cd /content/repo

# 1. Tải PromptIR All.ckpt và toàn bộ testsets (BSD68, Kodak24, Urban100, Rain100L)
!python scripts/download_datasets.py

# 2. Kiểm tra số lượng ảnh đạt chuẩn bài báo
!python scripts/verify_data_counts.py data/test

---
## BƯỚC 4: P1-T1 TÁI HIỆN BASELINE TRÊN CÁC TẬP KIỂM CHUẨN (TESTSETS)

In [ ]:
%cd /content/repo

# 1. Đánh giá PromptIR trên checkpoint All.ckpt
!python scripts/evaluate_baselines.py --baseline promptir --weights baselines/weights/promptir_all.ckpt --data_dir data/test

# 2. Đánh giá AdaIR nếu có checkpoint
import glob
adair_ckpts = glob.glob("baselines/weights/adair/*.pth") + glob.glob("baselines/weights/adair/*.pt")
if adair_ckpts:
    print(f"Phát hiện checkpoint AdaIR: {adair_ckpts[0]}")
    !python scripts/evaluate_baselines.py --baseline adair --weights {adair_ckpts[0]} --data_dir data/test

import pandas as pd
from pathlib import Path
csv_candidates = [Path("results/baseline_results.csv"), Path("repo/results/baseline_results.csv")]
csv_path = next((p for p in csv_candidates if p.exists()), None)
if csv_path:
    df_base = pd.read_csv(csv_path)
    print("\n=== KẾT QUẢ TÁI HIỆN BASELINE ===")
    display(df_base)
else:
    print("[-] Chưa tìm thấy file kết quả baseline_results.csv")

---
## BƯỚC 5: P1-T2 ĐO ĐẠC HIỆU NĂNG THỰC TẾ TRÊN GPU T4 (FLOPS, LATENCY, VRAM)

In [ ]:
%cd /content/repo

!python scripts/profile_efficiency.py

import pandas as pd
from pathlib import Path
csv_candidates = [Path("results/efficiency.csv"), Path("repo/results/efficiency.csv")]
csv_path = next((p for p in csv_candidates if p.exists()), None)
if csv_path:
    df_eff = pd.read_csv(csv_path)
    print("\n=== BẢNG ĐO HIỆU NĂNG PHẦN CỨNG T4 ===")
    display(df_eff)

---
## BƯỚC 6: P2-T2 & P2-T3 CHẨN ĐOÁN THẤT BẠI CHUYÊN SÂU & THỬ THÁCH OOD

In [ ]:
%cd /content/repo

!python scripts/diagnose_failures.py --baseline promptir --weights baselines/weights/promptir_all.ckpt --data_dir data/test --output_dir results/diagnostics --max_samples 50

import pandas as pd
from pathlib import Path
csv_candidates = [Path("results/ood_baseline.csv"), Path("repo/results/ood_baseline.csv")]
csv_path = next((p for p in csv_candidates if p.exists()), None)
if csv_path:
    df_ood = pd.read_csv(csv_path)
    print("\n=== KẾT QUẢ THỬ THÁCH SUY THOÁI CHƯA TỪNG GẶP (OOD) ===")
    display(df_ood)

# Hiển thị 3 ảnh chẩn đoán lỗi tiêu biểu
from PIL import Image
from IPython.display import display as ipy_display
worst_dirs = [Path("results/diagnostics/worst_cases"), Path("repo/results/diagnostics/worst_cases")]
worst_dir = next((d for d in worst_dirs if d.exists()), None)
if worst_dir:
    worst_imgs = sorted(list(worst_dir.glob("*.png")))[:3]
    for img_p in worst_imgs:
        print(f"\nMinh họa thất bại: {img_p.name} (Clean | Degraded | Restored | Error Heatmap | FFT Spectrum)")
        ipy_display(Image.open(img_p))

---
## BƯỚC 7: TỰ ĐỘNG ĐÓNG GÓI VÀ SAO LƯU KẾT QUẢ

In [ ]:
import os, datetime, zipfile, shutil
from pathlib import Path

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"aioir_full_results_{timestamp}.zip"
print(f"Đang nén kết quả vào {zip_filename}...")

with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for folder in [Path("results"), Path("analysis"), Path("repo/results"), Path("repo/analysis")]:
        if folder.exists():
            for f in folder.rglob("*"):
                if f.is_file():
                    zipf.write(f, str(f))

drive_backup_dir = Path("/content/drive/MyDrive/nckh_coChau_workspace/runs_backup")
if Path("/content/drive/MyDrive").exists():
    drive_backup_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(zip_filename, drive_backup_dir / zip_filename)
    print(f"\n[XÁC NHẬN] ĐÃ SAO LƯU KẾT QUẢ THÀNH CÔNG VÀO GOOGLE DRIVE:")
    print(f">> Đường dẫn: {drive_backup_dir / zip_filename}")
else:
    print(f"\n[XÁC NHẬN] Đã nén kết quả tại /content/repo/{zip_filename}.")
    print("Bạn có thể tải file này về máy tính bằng code: from google.colab import files; files.download('" + zip_filename + "')")

print("\nVui lòng copy toàn bộ text đầu ra của notebook này và gửi cho AI cộng sự để tiến hành Phase P3!")